# **HydroServer Example 4: Custom Python ETL Pipelines for Uploading GEOGLOWS Streamflow Forecasts to HydroServer**

### **Overview**

This exercise demonstrates how to create a custom **Extract, Transform, and Load (ETL) pipeline** to automatically retrieve streamflow forecasts from the **GEOGLOWS River Forecast System (RFS)** and upload them to HydroServer.

The example uses a **GEOGLOWS river reach on the Nzoia River near Webuye, Kenya**, and retrieves the latest available streamflow forecast through the GEOGLOWS API.

### **Description**

This Jupyter Notebook demonstrates how to use `hydroserverpy` to create the required HydroServer resources and configure an automated ETL workflow for forecast data.

The exercise demonstrates how to:

1. Connect to **HydroServer**.
2. Create a **Workspace**.
3. Create a monitoring site on the **Nzoia River near Webuye, Kenya**.
4. Create the **Streamflow Observed Property** and its unit (**m³/s**).
5. Create a **GEOGLOWS model/sensor** and Processing Level.
6. Create a **Datastream** for forecasted streamflow.
7. Configure an **HTTP Extractor** to retrieve the latest forecast from the **GEOGLOWS API**.
8. Configure a **CSV Transformer** to transform the GEOGLOWS forecast into a format compatible with HydroServer.
9. Configure a **HydroServer Loader** to upload the forecast values to the Datastream.
10. Run the complete **ETL pipeline** to automatically extract, transform, and load the latest GEOGLOWS streamflow forecast into HydroServer.

For this example, we use **GEOGLOWS River ID `160193736`**, corresponding to a modeled river reach near the Nzoia River monitoring location at Webuye, Kenya.

### **GEOGLOWS**

**GEOGLOWS** stands for **Group on Earth Observations Global Water Sustainability**.

1. Provides **global streamflow forecasts and historical simulations**.
2. Uses **ECMWF meteorological forecasts** to generate hydrological forecasts.
3. Divides the river network into **river reaches**, each identified by a unique **River ID**.
4. Provides **retrospective and near-real-time forecast data**.
5. Data can be accessed through the **GEOGLOWS API** for automated workflows.
6. This allows us to **automatically retrieve the latest forecasts and ingest them into HydroServer using an HTTP/ETL workflow**.

### **Prerequisites**

You must have an account on the **HydroServer Playground** instance to run this notebook. If you have not created an account yet, navigate to:

https://playground.hydroserver.org

and follow the instructions to create a new user account.

### **Software Requirements**

This notebook uses **Python** and the `hydroserverpy` Python package to interact with HydroServer and configure the ETL pipeline.

More detailed examples and documentation are available at:

* https://hydroserver2.github.io/hydroserver/user-guides/how-to/using-the-python-client.html
* https://www.hydroserver.org
* https://geoglows.ecmwf.int/

### **Import Required Python Packages**

First, import the Python libraries required for this exercise.

These packages will be used to:

- Work with **dates and times** for forecast timestamps.
- Securely enter the **HydroServer password**.
- Retrieve data from the **GEOGLOWS API** using HTTP requests.
- Organize and manipulate forecast data with **pandas**.
- Visualize streamflow forecasts using **Matplotlib**.
- Connect to and interact with **HydroServer** using `hydroserverpy`.

In [1]:
%pip install -q hydroserverpy==1.11.3 pandas requests matplotlib
from datetime import datetime, timedelta, timezone
from getpass import getpass
import matplotlib.pyplot as plt
import pandas as pd
import requests
from hydroserverpy import HydroServer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.4/113.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 10.8 MB/s eta 0:00:00


### **Import HydroServer ETL Components**

Next, import the `hydroserverpy` components required to build the custom **Extract, Transform, and Load (ETL) pipeline**.

These components will be used to:

- **Extract** the latest streamflow forecast from the **GEOGLOWS API** using `HTTPExtractor`.
- **Transform** the retrieved data into the structure required by HydroServer using `CSVTransformer` and data mappings.
- **Load** the transformed forecast data into a HydroServer Datastream using `HydroServerLoader`.
- Combine these steps into a complete automated workflow using `ETLPipeline`.

In [2]:
from hydroserverpy.etl import ETLPipeline
from hydroserverpy.etl.extractors import HTTPExtractor
from hydroserverpy.etl.transformers import (
    CSVTransformer,
    JSONTransformer,
    ETLDataMapping,
    ETLTargetPath
)
from hydroserverpy.etl.operations import ArithmeticExpressionOperation
from hydroserverpy.etl.loaders import HydroServerLoader

### **Set the Initial Parameters to Connect to HydroServer**

The first step in interacting with a HydroServer instance is to create a connection to that instance. For this example, we will use your username and password because we will create the Workspace in code.

**IMPORTANT: In the following code, change the email to match the HydroServer user account you created.**

In [4]:
# Set initial parameters to connect to HydroServer
hydroserver_host = 'https://playground.hydroserver.org'

# Change the email and password below to your HydroServer username and password
hydroserver_email = 'svicario@lincolninst.edu' #'user@youremail.com'
hydroserver_password = getpass('Enter your HydroServer password: ') #getpass('Enter your HydroServer password: ')

Enter your HydroServer password: ··········


### **Connect to HydroServer Using Environment Variables (OPTION 2)**

For an automated ETL workflow, the HydroServer credentials should not require manual input each time the script runs.

Store your HydroServer email and password as Windows environment variables named `HYDROSERVER_EMAIL` and `HYDROSERVER_PASSWORD`. The Python script can then retrieve these credentials automatically when it is executed by Windows Task Scheduler.

In [5]:
#import os

# Retrieve HydroServer credentials from Windows environment variables
#hydroserver_email = os.environ["HYDROSERVER_EMAIL"]
#hydroserver_password = os.environ["HYDROSERVER_PASSWORD"]

# Connect to HydroServer
#hydroserver_host = "https://playground.hydroserver.org"

#hs = HydroServer(
#    host=hydroserver_host,
#    email=hydroserver_email,
#    password=hydroserver_password
#)

#print("Successfully connected to HydroServer using environment variables.")

### **Initialize HydroServer Connection**

Initialize the connection to HydroServer with the connection information specified above.

In [6]:
# Initialize HydroServer connection with credentials.
hs = HydroServer(
    host=hydroserver_host,
    email=hydroserver_email,
    password=hydroserver_password
)

print('\nSuccessfully connected to HydroServer!')


Successfully connected to HydroServer!


### **Connect to Your Existing Workspace**

In the previous exercises, you created a HydroServer Workspace for the training. In this exercise, we will **reuse that existing Workspace** rather than create a new one.

Specify the name of your existing Workspace and retrieve its information from HydroServer. The Workspace ID will be used to create the resources needed for the GEOGLOWS forecast workflow.

In [7]:
# Enter the name of the workspace created in the previous exercise
workspace_name = "Rwanda Training 2026 - Sara"

# Retrieve workspaces associated with your account
workspaces = hs.workspaces.list(is_associated=True)

# Find the workspace by name
workspace = next(
    ws for ws in workspaces.items
    if ws.name == workspace_name
)

# Save the Workspace ID for later use
workspace_id = workspace.uid

print(f"Using workspace: {workspace.name}")
print(f"Workspace ID: {workspace_id}")

Using workspace: Rwanda Training 2026 - Sara
Workspace ID: 01a062cb-7ae3-7dd6-a6d6-0ea739c93849


### **Create the Nzoia River Monitoring Site**

Next, create a **monitoring site (Thing)** in HydroServer representing the Nzoia River near **Webuye, Kenya**.

For this exercise, we use the location of gauge **1DA02**. Its coordinates are also used to identify the nearest GEOGLOWS modeled river reach, **River ID `160193736`**, from which we will retrieve the streamflow forecasts.

The Monitoring Site ID is saved because it will be needed when creating the forecast Datastream.

In [8]:
# Create the monitoring site for the Nzoia River near Webuye, Kenya
# Gauge/reference station: 1DA02
# Coordinates used to identify the nearest GEOGLOWS river reach
nzoia_station = hs.things.create(
    workspace=workspace_id,
    name="Nzoia River near Webuye",
    description=(
        "Reference location for gauge 1DA02 on the Nzoia River near Webuye, Kenya. "
        "The nearest GEOGLOWS V2 modeled river reach is 160193736."
    ),
    sampling_feature_type="Site",
    sampling_feature_code="1DA02",
    site_type="Stream",
    latitude=0.58595,
    longitude=34.80685,
    country="KE",
    is_private=False
)

thing_id = nzoia_station.uid

print("Created monitoring site:")
print(f"{nzoia_station.name}: {thing_id}")


Created monitoring site:
Nzoia River near Webuye: 01a06812-6cda-7ea1-8a63-92f013512b7d


### **Create the GEOGLOWS Model Metadata**

Next, create a **Sensor** in HydroServer to describe the source of the forecast data.

In this exercise, the Sensor represents the **GEOGLOWS River Forecast System (RFS) V2**, which provides the modeled streamflow forecasts that will be retrieved through the GEOGLOWS API.

The Sensor ID will later be associated with the forecast Datastream.

In [9]:
# Create metadata describing GEOGLOWS as the source/model
geoglows_sensor = hs.sensors.create(
    workspace=workspace_id,
    name="GEOGLOWS RFS V2",
    description=(
        "Modeled streamflow forecasts from the GEOGLOWS River Forecast System V2 "
        "for river reach 160193736."
    ),
    encoding_type="text/csv",
    method_type="Model Simulation",
    method_code="geoglows-rfs-v2"
)

print("Created sensor/model metadata:")
print(f"{geoglows_sensor.name}: {geoglows_sensor.uid}")


Created sensor/model metadata:
GEOGLOWS RFS V2: 01a06812-59dd-7ca6-8b8d-c1ce8a37ddd2


### **Create the Streamflow Observed Property**

Next, create the **Observed Property** that defines the variable represented by the forecast data.

For this exercise, the Observed Property is **Streamflow**, representing the forecasted river discharge provided by GEOGLOWS.

The Observed Property ID will later be associated with the forecast Datastream.

In [10]:
# Create the observed property
streamflow = hs.observedproperties.create(
    workspace=workspace_id,
    name="Streamflow",
    definition="Streamflow",
    description="Forecasted river discharge (streamflow).",
    observed_property_type="Hydrology",
    code="Streamflow"
)

print("Created observed property:")
print(f"{streamflow.name}: {streamflow.uid}")


Created observed property:
Streamflow: 01a06812-9470-7cc3-b971-acf5b144d52c


### **Create the Streamflow Unit**

Next, create the **Unit** associated with the Streamflow Observed Property.

GEOGLOWS reports streamflow forecasts in **cubic meters per second (m³/s)**. This unit will later be associated with the forecast Datastream.

In [11]:
# GEOGLOWS streamflow is reported in cubic meters per second
streamflow_unit = hs.units.create(
    workspace=workspace_id,
    name="Cubic meter per second",
    symbol="m^3/s",
    definition="Cubic meters per second",
    unit_type="Volumetric Flow Rate"
)

print("Created unit:")
print(f"{streamflow_unit.name}: {streamflow_unit.uid}")


Created unit:
Cubic meter per second: 01a06812-9472-73bc-b5df-cfdaf7e6e4a7


### **Create the Forecast Processing Level**

Next, create a **Processing Level** to indicate that the streamflow values are **forecasted model outputs** rather than observed measurements.

For this exercise, the Processing Level identifies the data as streamflow forecasts produced by **GEOGLOWS RFS V2**. It will later be associated with the forecast Datastream.

In [12]:
# Processing level for forecast/model output
forecast_processing = hs.processinglevels.create(
    workspace=workspace_id,
    code="Forecast",
    definition="Forecast streamflow",
    explanation="Streamflow forecast produced by GEOGLOWS RFS V2."
)

print("Created processing level:")
print(f"{forecast_processing.code}: {forecast_processing.uid}")


Created processing level:
Forecast: 01a06812-9474-73bc-9f5e-0967c338aa8b


### **Create the GEOGLOWS Forecast Datastream**

Next, create the **Datastream** that will store the GEOGLOWS streamflow forecast in HydroServer.

The Datastream connects the **Nzoia River monitoring site** with the GEOGLOWS model, Streamflow Observed Property, unit, and Forecast Processing Level created in the previous steps.

Because GEOGLOWS provides forecasted streamflow at **3-hour intervals**, the Datastream is configured with a 3-hour time spacing and a forecast period extending up to **15 days ahead**.

In [13]:
forecast_datastream = hs.datastreams.create(
    name=f"GEOGLOWS Streamflow Forecast - {nzoia_station.name}",
    description=(
        "Latest GEOGLOWS RFS V2 streamflow forecast for river reach 160193736, "
        "near gauge 1DA02 on the Nzoia River."
    ),
    thing=nzoia_station.uid,
    sensor=geoglows_sensor.uid,
    observed_property=streamflow.uid,
    processing_level=forecast_processing.uid,
    unit=streamflow_unit.uid,
    observation_type="Model Simulation",
    result_type="Timeseries",
    sampled_medium="Surface Water",
    no_data_value=-9999,
    aggregation_statistic="Average",
    time_aggregation_interval=3,
    time_aggregation_interval_unit="hours",
    intended_time_spacing=3,
    intended_time_spacing_unit="hours",
    status="Ongoing",
    is_private=False,
    is_visible=True
)

### **Step 1: Extract – Connect to the GEOGLOWS API**

The first step of the ETL pipeline is to **extract the forecast data from its source**.

For this exercise, we use the GEOGLOWS V2 River ID **`160193736`**, representing the modeled river reach nearest to gauge **1DA02** on the Nzoia River.

The `HTTPExtractor` connects to the **GEOGLOWS API** and retrieves the latest available streamflow forecast in **CSV format**. The forecast will later be transformed and loaded into the HydroServer Datastream.

In [14]:
# GEOGLOWS V2 river reach nearest gauge 1DA02 on the Nzoia River
GEOGLOWS_RIVER_ID = 160193736

forecast_url = (
    f"https://geoglows.ecmwf.int/api/v2/"
    f"forecast/{GEOGLOWS_RIVER_ID}?format=csv"
)

extractor = HTTPExtractor(source_uri=forecast_url)

print("Extractor created.")
print(f"Source: {forecast_url}")


Extractor created.
Source: https://geoglows.ecmwf.int/api/v2/forecast/160193736?format=csv


### **Step 2: Transform – Parse the GEOGLOWS Forecast Data**

The second step of the ETL pipeline is to **transform the extracted forecast data** into a structure that can be loaded into HydroServer.

The GEOGLOWS API returns the forecast as a CSV file. The `CSVTransformer` reads this file, identifies the **`datetime`** column as the forecast timestamp, and parses the rows containing the forecast values.

In the next step, we will define how the GEOGLOWS streamflow values are mapped to the HydroServer Datastream.

In [15]:
# Parse the GEOGLOWS CSV response.
# The API uses a datetime column for forecast timestamps.
transformer = CSVTransformer(
    timestamp_key="datetime",
    delimiter=",",
    header_row=1,
    data_start_row=2
)

print("Transformer created.")

Transformer created.


### **Step 3: Data Mapping – Map the Forecast Values**

Next, define how the GEOGLOWS forecast values should be mapped to the HydroServer Datastream.

The GEOGLOWS API returns several forecast fields. For this exercise, we use **`flow_median`**, which represents the median streamflow forecast.

The `ETLDataMapping` links this field to the GEOGLOWS forecast Datastream created earlier, telling the ETL pipeline where the forecast values should be stored in HydroServer.

In [16]:
# Map the GEOGLOWS forecast values to the HydroServer forecast datastream.
# `flow_median` is the forecast series returned by this GEOGLOWS endpoint.
data_mappings = [
    ETLDataMapping(
        source_identifier="flow_median",
        target_paths=[
            ETLTargetPath(
                target_identifier=str(forecast_datastream.uid)
            )
        ]
    )
]

print("Data mapping created.")

Data mapping created.


### **Step 3: Load – Configure the HydroServer Loader**

The final step of the ETL pipeline is to **load the transformed forecast data into HydroServer**.

The `HydroServerLoader` uses the authenticated HydroServer connection created earlier to upload the GEOGLOWS forecast values to the target Datastream.

The `chunk_size` parameter controls how many observations are uploaded at a time.

In [17]:
loader = HydroServerLoader(
    client=hs,
    chunk_size=5000
)

print("Loader created.")

Loader created.


### **Assemble the ETL Pipeline**

Now that the **Extractor**, **Transformer**, and **Loader** have been configured, combine them into a single `ETLPipeline`.

The complete pipeline defines the workflow that will:

1. **Extract** the latest streamflow forecast from the GEOGLOWS API.
2. **Transform** the CSV forecast data into the required structure.
3. **Load** the forecast values into the HydroServer Datastream.

The pipeline is now ready to run.

In [19]:
# Assemble the ETL pipeline
pipeline = ETLPipeline(
    extractor=extractor,
    transformer=transformer,
    loader=loader
)

print("ETL pipeline created.")

ETL pipeline created.


### **Run the ETL Pipeline**

Finally, run the complete ETL pipeline to automatically retrieve and upload the latest GEOGLOWS streamflow forecast.

When the pipeline runs, it will:

1. **Extract** the latest forecast from the GEOGLOWS API.
2. **Transform** the CSV response and identify the forecast timestamps and `flow_median` values.
3. **Map** the forecast values to the HydroServer Datastream.
4. **Load** the forecast observations into HydroServer.

The pipeline status and final execution stage are displayed after the run. If the pipeline completes successfully, the latest GEOGLOWS streamflow forecast will be available in the HydroServer Datastream.

In [20]:
# Run the pipeline: retrieve the latest GEOGLOWS forecast and upload it to HydroServer
context = pipeline.run(
    data_mappings=data_mappings,
    raise_on_error=False
)

print(f"Pipeline status: {context.status}")
print(f"Final stage: {context.stage}")

if context.results is not None:
    print(context.results)

# If the run is successful, the latest GEOGLOWS forecast values are now
# stored in the HydroServer forecast datastream created above.

INFO:hydroserverpy.etl.models.pipeline:Starting extract
INFO:hydroserverpy.etl.models.pipeline:Resolved runtime source URI: https://geoglows.ecmwf.int/api/v2/forecast/160193736?format=csv
INFO:hydroserverpy.etl.extractors.http:Requesting data from source URI
INFO:hydroserverpy.etl.models.pipeline:Extractor returned payload: {'type': 'BytesIO', 'bytes': 4627}
INFO:hydroserverpy.etl.models.pipeline:Starting transform
INFO:hydroserverpy.etl.models.pipeline:Transform result: {'type': 'DataFrame', 'rows': 120, 'columns': 2, 'datastreams': 1}
INFO:hydroserverpy.etl.models.pipeline:Starting load
INFO:hydroserverpy.etl.loaders.hydroserver:Load result: loaded=120 available=120 cutoff=None


Pipeline status: SUCCESS
Final stage: CLEANUP
success_count=1 failure_count=0 skipped_count=0 values_loaded_total=120 earliest_timestamp=None latest_timestamp=None target_results={'01a06812-7445-7514-aa7f-3336b07481d8': ETLTargetResult(target_identifier='01a06812-7445-7514-aa7f-3336b07481d8', status='success', values_loaded=120, earliest_timestamp=None, latest_timestamp=None, error=None, traceback=None)}
